# 03 — HeteroGAT (Epic 3)

**Two tasks (multi-task head):**
- **Node-level**: predict per-node criticality (`data['compute'].y`)  → Spearman ρ
- **Graph-level**: predict total step time (`data.y`) → MSE + ΔT%

**Two model variants:**
- **Bidirectional** — standard GAT, messages flow both directions (associational baseline)
- **Directed** — messages flow ancestor → descendant only (causal inductive bias)

**Acceptance criteria (Sprint 5):**
- Directed variant: Spearman ρ > 0.70 on val set
- Both variants: ΔT% < 5% on val set
- Both val loss curves decreasing

**Data:** `preethamvj/bottleneck-oracle-graphs` on HuggingFace (501 graphs)

**No MLP/baseline comparison in this notebook** — baselines are in `model_1f1b.ipynb`.


In [1]:
!pip install -q torch-geometric huggingface_hub scikit-learn scipy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

# The -q flag keeps the output quiet so it doesn't flood your notebook
!unzip -q /content/drive/MyDrive/BottleneckOracle_Backup/data_backup.zip -d /content/

Mounted at /content/drive


The above will recreate the /content/data/raw and /content/data/graphs folders.

In [3]:
import os, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, GATConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
from huggingface_hub import list_repo_files, hf_hub_download
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

os.makedirs('data', exist_ok=True)


Device: cuda


## 1. Load graphs from local drive

In [4]:
import torch
import glob
import os

# Grab all the graph files generated by Notebook 4
graph_paths = glob.glob('/content/content/data/graphs/*.pt')

# Load them directly into a list for PyTorch Geometric
graphs = [torch.load(path, weights_only=False) for path in graph_paths]

print(f"Successfully loaded {len(graphs)} local graphs!")

Successfully loaded 501 local graphs!


In [5]:
# # ── quick sanity on one graph ─────────────────────────────────────────────────

g0 = graphs[0]
assert g0['compute'].x.shape[1] == 4, "Expected 4 compute features"
assert g0['compute'].y is not None,    "Missing node labels"
assert g0.y is not None,               "Missing graph label"
print(f"Graph schema: {g0}")
print(f"Compute nodes: {g0['compute'].x.shape[0]}")
print(f"Edge types: {g0.edge_types}")

Graph schema: HeteroData(
  y=[1],
  compute={
    x=[1486, 4],
    y=[1486],
    y_binary=[1486],
  },
  (compute, depends_on, compute)={ edge_index=[2, 1101257] }
)
Compute nodes: 1486
Edge types: [('compute', 'depends_on', 'compute')]


## 2. Train / Val / Test split

**Same SEED and 70/15/15 ratios as `model_1f1b.ipynb`** so the held-out test set is identical
across all notebooks.  If `data/splits.pt` exists (written by `model_1f1b`) we load it;
otherwise we recompute with the same seed.


In [6]:
SPLITS_PATH = 'data/splits.pt'

if os.path.exists(SPLITS_PATH):
    splits   = torch.load(SPLITS_PATH)
    train_idx = splits['train']
    val_idx   = splits['val']
    test_idx  = splits['test']
    print('Loaded splits from data/splits.pt')
else:
    n = len(graphs)
    idx = list(range(n))
    train_idx, temp_idx = train_test_split(idx, test_size=0.30, random_state=SEED)
    val_idx,   test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED)
    torch.save({'train': train_idx, 'val': val_idx, 'test': test_idx}, SPLITS_PATH)
    print('Splits recomputed and saved.')

train_graphs = [graphs[i] for i in train_idx]
val_graphs   = [graphs[i] for i in val_idx]
test_graphs  = [graphs[i] for i in test_idx]
print(f"Split → train:{len(train_graphs)}  val:{len(val_graphs)}  test:{len(test_graphs)}")


Splits recomputed and saved.
Split → train:350  val:75  test:76


## 3. HeteroGAT model

### Node ordering contract (IMPORTANT — read before editing)

Graphs from `02-trace-generation-fixed.ipynb` assign node IDs by timestamp order
(`ops = sorted(ops, key=lambda x: x['ts'])`), and all edges are `i-1 → i`.
`compute_idx` is built from `G.nodes()` iteration, which preserves insertion
(= timestamp) order.

**Consequence:** for every `('compute','depends_on','compute')` edge `(src, dst)`,
`src < dst` is guaranteed by construction.

The directed variant exploits this by masking `edge_index` to keep only `src < dst`.
If `02` ever changes to add non-sequential edges or reindex nodes differently,
this assumption will silently break.  The assertion in `_safe_directed_mask` will
catch that at runtime.

### Architecture

- 2-layer HeteroConv with GATConv kernels
- Edge types used: `compute→compute` (depends_on), `compute→network` (sends_to),
  `network→compute` (feeds)
- Node head: per-compute-node criticality logit → BCE loss
- Graph head: mean-pool compute embeddings → step-time regression → MSE loss


In [7]:
# OPTIONAL: CriticalPathLayer for topological slack propagation
# This layer mimics CPM via message passing for better OOD generalization

class CriticalPathLayer(torch.nn.Module):
    """
    Message-passing layer that approximates Critical Path Method (CPM).

    Computes node slack values by propagating timing information along
    the graph topology, similar to how CPM computes ES/EF/LS/LF.

    This provides an inductive bias that matches the ground-truth label
    computation process, improving OOD generalization.
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        # Learnable weights for combining forward/backward passes
        self.forward_weight = torch.nn.Parameter(torch.tensor(0.5))
        self.backward_weight = torch.nn.Parameter(torch.tensor(0.5))

        # Projection for duration to hidden space
        self.duration_proj = torch.nn.Linear(1, hidden_dim)

    def forward(self, x_dict, edge_index_dict):
        """
        Args:
            x_dict: Node features (includes duration as first feature)
            edge_index_dict: Edge indices for message passing

        Returns:
            Updated node features with CPM-inspired timing information
        """
        compute_x = x_dict['compute']  # [N, hidden_dim]
        durations = compute_x[:, 0:1]  # [N, 1] - duration is first feature

        # Get compute->compute edges
        cc_edges = edge_index_dict.get(('compute', 'depends_on', 'compute'))
        if cc_edges is None or cc_edges.shape[1] == 0:
            return x_dict  # No edges, return original features

        # Build adjacency for efficient message passing
        num_nodes = compute_x.shape[0]
        edge_index = cc_edges  # [2, E]

        # Forward pass: compute earliest start (ES) - max of predecessors' EF
        # ES[i] = max(ES[j] + duration[j]) for all predecessors j of i
        row, col = edge_index  # row=source, col=target

        # Initialize ES with zeros
        es = torch.zeros(num_nodes, device=compute_x.device)
        ef = durations.squeeze()  # EF = ES + duration (initially)

        # Iteratively propagate forward until convergence
        for _ in range(edge_index.shape[1]):  # Max iterations = path length
            new_es = torch.zeros_like(es)
            for i in range(num_nodes):
                # Find all predecessors of i
                predecessors = col[row == i]
                if predecessors.numel() > 0:
                    new_es[i] = torch.max(ef[predecessors])
                else:
                    new_es[i] = 0.0  # Source nodes
            ef = new_es + durations.squeeze()
            if torch.allclose(es, new_es, atol=1e-6):
                break
            es = new_es

        # Backward pass: compute latest start (LS) - min of successors' LS
        # LS[i] = min(LS[j] - duration[i]) for all successors j of i
        project_end = torch.max(ef)
        ls = project_end - durations.squeeze()  # Initialize with project end

        # Iteratively propagate backward
        for _ in range(edge_index.shape[1]):
            new_ls = project_end - durations.squeeze()
            for i in range(num_nodes):
                # Find all successors of i
                successors = row[col == i]
                if successors.numel() > 0:
                    new_ls[i] = torch.min(ls[successors]) - durations[i]
            if torch.allclose(ls, new_ls, atol=1e-6):
                break
            ls = new_ls

        # Slack = LS - ES
        slack = torch.clamp(ls - es, min=0.0)  # Ensure non-negative

        # Add slack information to node features
        slack_features = self.duration_proj(slack.unsqueeze(-1))
        updated_x = compute_x + slack_features

        # Update compute node features
        x_dict['compute'] = updated_x

        return x_dict

print("CriticalPathLayer defined (optional, for improved OOD generalization)")

CriticalPathLayer defined (optional, for improved OOD generalization)


In [8]:
class HeteroGAT(nn.Module):
    """
    Two-layer heterogeneous GAT with slack regression.

    Outputs:
      node_slack : [N_compute]  — predicted slack (continuous, in ms)
      step_time   : scalar       — predicted total step time (ms)

    directed=True  → only forward edges (src < dst) passed to conv layers
    directed=False → standard bidirectional GAT

    CHANGE: Now predicts continuous slack instead of binary criticality.
    """
    def __init__(self, in_dim: int = 4, hidden: int = 64,
                 heads: int = 4, directed: bool = False):
        super().__init__()
        self.directed = directed

        # ── Layer 1 ───────────────────────────────────────────────────────────
        self.conv1 = HeteroConv({
            ('compute', 'depends_on', 'compute'):
                GATConv(in_dim, hidden // heads, heads=heads, add_self_loops=False),
            ('compute', 'sends_to', 'network'):
                GATConv(in_dim, hidden // heads, heads=heads, add_self_loops=False),
            ('network', 'feeds', 'compute'):
                GATConv(in_dim, hidden // heads, heads=heads, add_self_loops=False),
        }, aggr='sum')

        # ── Layer 2 ───────────────────────────────────────────────────────────
        self.conv2 = HeteroConv({
            ('compute', 'depends_on', 'compute'):
                GATConv(hidden, hidden // heads, heads=heads, add_self_loops=False),
            ('compute', 'sends_to', 'network'):
                GATConv(hidden, hidden // heads, heads=heads, add_self_loops=False),
            ('network', 'feeds', 'compute'):
                GATConv(hidden, hidden // heads, heads=heads, add_self_loops=False),
        }, aggr='sum')

        # ── heads ─────────────────────────────────────────────────────────────
        # Changed from binary classification to regression for slack prediction
        self.node_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1)  # Single output for slack regression
        )
        self.graph_head = nn.Sequential(
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )

    # ── directed masking ──────────────────────────────────────────────────────
    @staticmethod
    def _safe_directed_mask(edge_index: torch.Tensor) -> torch.Tensor:
        """
        Keep only forward edges (src < dst).

        Relies on the src<dst guarantee from 02-trace-generation-fixed.ipynb.
        Asserts that at least *some* edges satisfy src<dst (catches reindexing
        regressions immediately rather than silently training on empty graphs).
        """
        if edge_index is None or edge_index.shape[1] == 0:
            return edge_index
        mask = edge_index[0] < edge_index[1]
        assert mask.any(), (
            "Directed masking produced 0 forward edges.  "
            "Check whether 02 still generates timestamp-ordered node IDs. "
            "If graph construction changed, update the masking strategy."
        )
        return edge_index[:, mask]

    def _apply_directed(self, edge_index_dict):
        out = {}
        for k, ei in edge_index_dict.items():
            if ei is not None and ei.shape[1] > 0:
                out[k] = self._safe_directed_mask(ei)
            else:
                out[k] = ei
        return out

    def forward(self, x_dict, edge_index_dict):
        if self.directed:
            edge_index_dict = self._apply_directed(edge_index_dict)

        # ── layer 1 ───────────────────────────────────────────────────────────
        # Only pass edge types that actually exist in this graph
        present = {k: v for k, v in edge_index_dict.items()
                   if v is not None and v.shape[1] > 0}
        h = self.conv1(x_dict, present)
        h = {k: F.elu(v) for k, v in h.items()}
        # carry forward node types not updated by conv (missing in some graphs)
        for k in x_dict:
            if k not in h:
                h[k] = x_dict[k]

        # ── layer 2 ───────────────────────────────────────────────────────────
        present2 = {k: v for k, v in edge_index_dict.items()
                    if v is not None and v.shape[1] > 0}
        h2 = self.conv2(h, present2)
        h2 = {k: F.elu(v) for k, v in h2.items()}
        for k in h:
            if k not in h2:
                h2[k] = h[k]

        compute_emb = h2['compute']                              # [N_c, hidden]
        node_slack = self.node_head(compute_emb).squeeze(-1)     # [N_c] - continuous slack
        graph_emb   = compute_emb.mean(dim=0, keepdim=True)      # [1, hidden]
        step_time   = self.graph_head(graph_emb).squeeze()       # scalar

        return node_slack, step_time

## 4. Training utilities

In [9]:
def delta_t_pct(pred, true):
    """Absolute % error in step-time prediction."""
    return abs(pred - true) / (true + 1e-8) * 100


def train_one_epoch(model, graphs, optimizer, node_weight=0.5, clip_grad=1.0):
    model.train()
    total_loss = 0.0
    for g in graphs:
        g = g.to(DEVICE)
        optimizer.zero_grad()

        node_slack, step_pred = model(g.x_dict, g.edge_index_dict)

        # graph-level: MSE on step time
        graph_loss = F.mse_loss(step_pred, g.y.squeeze().to(DEVICE))

        # node-level: MSE/Huber on slack (changed from BCE)
        node_labels = g['compute'].y.to(DEVICE)  # Continuous slack labels
        node_loss   = F.mse_loss(node_slack, node_labels)  # Changed to MSE

        loss = (1 - node_weight) * graph_loss + node_weight * node_loss
        loss.backward()

        # gradient clipping — prevents rare exploding-gradient spikes in GAT
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(graphs)


@torch.no_grad()
def evaluate(model, graphs):
    model.eval()
    pred_times, true_times = [], []
    node_rho_list = []
    slack_mse_list = []
    slack_mae_list = []

    for g in graphs:
        g = g.to(DEVICE)
        node_slack, step_pred = model(g.x_dict, g.edge_index_dict)

        pred_times.append(step_pred.item())
        true_times.append(g.y.item())

        # Slack prediction metrics
        true_slack = g['compute'].y.cpu().numpy()
        pred_slack = node_slack.cpu().numpy()

        # Skip degenerate graphs where all labels are identical
        if true_slack.std() > 0:
            slack_mse = np.mean((pred_slack - true_slack) ** 2)
            slack_mae = np.mean(np.abs(pred_slack - true_slack))
            slack_mse_list.append(slack_mse)
            slack_mae_list.append(slack_mae)

            # Also compute Spearman on slack for reference
            rho, _ = spearmanr(pred_slack, true_slack)
            node_rho_list.append(rho)

    mse    = mean_squared_error(true_times, pred_times)
    deltas = [delta_t_pct(p, t) for p, t in zip(pred_times, true_times)]
    spear  = float(np.mean(node_rho_list)) if node_rho_list else float('nan')

    return {
        'mse':             mse,
        'mean_dt':         float(np.mean(deltas)),
        'median_dt':       float(np.median(deltas)),
        'spearman':        spear,
        'slack_mse':       float(np.mean(slack_mse_list)) if slack_mse_list else float('nan'),
        'slack_mae':       float(np.mean(slack_mae_list)) if slack_mae_list else float('nan'),
    }

## 5. Train — Bidirectional variant

Checkpoint saved to `data/heterogat_bidir_best.pt` on val-MSE improvement.


In [10]:
import shutil

EPOCHS  = 80
LR      = 3e-4
NODE_W  = 0.5
CKPT_BIDIR = 'data/heterogat_bidir_best.pt'

# NEW: paths
local_save_path_bidir = '/content/best_heterogat_bidir.pt'
drive_save_path_bidir = '/content/drive/MyDrive/BottleneckOracle_Backup/best_heterogat_bidir.pt'

model_bidir = HeteroGAT(in_dim=4, hidden=64, heads=4, directed=False).to(DEVICE)
opt_bidir   = torch.optim.Adam(model_bidir.parameters(), lr=LR)

best_val_mse_bidir = float('inf')
print(f"{'Epoch':>6}  {'Train L':>9}  {'Val MSE':>9}  {'Val ΔT%':>8}  {'Slack MSE':>10}  {'Slack MAE':>10}  {'Val ρ':>7}  {'Saved':>5}")
print('-' * 90)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model_bidir, train_graphs, opt_bidir, node_weight=NODE_W)

    if epoch % 10 == 0:
        m = evaluate(model_bidir, val_graphs)
        saved = ''
        if m['mse'] < best_val_mse_bidir:
            best_val_mse_bidir = m['mse']

            # ORIGINAL SAVE (unchanged)
            torch.save(model_bidir.state_dict(), CKPT_BIDIR)

            # NEW: local fast save (full checkpoint)
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_bidir.state_dict(),
                'optimizer_state_dict': opt_bidir.state_dict(),
                'loss': best_val_mse_bidir,
            }, local_save_path_bidir)

            saved = '✅'

        print(f"{epoch:>6}  {train_loss:>9.4f}  {m['mse']:>9.2f}  "
              f"{m['mean_dt']:>7.2f}%  {m['slack_mse']:>10.4f}  {m['slack_mae']:>10.4f}  "
              f"{m['spearman']:>7.4f}  {saved}")

print(f'\n✅ Bidir done. Best val MSE: {best_val_mse_bidir:.4f}')

# NEW: one-time copy to Drive
shutil.copy(local_save_path_bidir, drive_save_path_bidir)
print("✅ Bidir model copied to Drive!")

 Epoch    Train L    Val MSE   Val ΔT%   Slack MSE   Slack MAE    Val ρ  Saved
------------------------------------------------------------------------------------------
    10    59.2674     121.42    29.69%      0.0036      0.0166   0.0594  ✅
    20    46.9380      68.51    21.25%      0.0047      0.0367  -0.0246  ✅
    30    10.0608      34.56     6.75%      0.0038      0.0197   0.0296  ✅
    40     6.5302       8.00     5.40%      0.0036      0.0179   0.0193  ✅
    50     6.0098       6.32     4.97%      0.0035      0.0149   0.0139  ✅
    60     5.6039       6.47     5.12%      0.0036      0.0147   0.0473  
    70     5.0675       6.47     5.14%      0.0035      0.0148   0.0793  
    80     4.7685       6.22     4.15%      0.0036      0.0149   0.0753  ✅

✅ Bidir done. Best val MSE: 6.2215
✅ Bidir model copied to Drive!


## 6. Train — Directed variant

Messages flow ancestor → descendant only (causal inductive bias).
See §3 for the `src < dst` ordering contract.


In [11]:
import shutil

CKPT_DIR = 'data/heterogat_dir_best.pt'

# NEW: paths
local_save_path_dir = '/content/best_heterogat_dir.pt'
drive_save_path_dir = '/content/drive/MyDrive/BottleneckOracle_Backup/best_heterogat_dir.pt'

model_dir = HeteroGAT(in_dim=4, hidden=64, heads=4, directed=True).to(DEVICE)
opt_dir   = torch.optim.Adam(model_dir.parameters(), lr=LR)

best_val_mse_dir = float('inf')
print(f"{'Epoch':>6}  {'Train L':>9}  {'Val MSE':>9}  {'Val ΔT%':>8}  {'Slack MSE':>10}  {'Slack MAE':>10}  {'Val ρ':>7}  {'Saved':>5}")
print('-' * 90)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model_dir, train_graphs, opt_dir, node_weight=NODE_W)

    if epoch % 10 == 0:
        m = evaluate(model_dir, val_graphs)
        saved = ''
        if m['mse'] < best_val_mse_dir:
            best_val_mse_dir = m['mse']

            # ORIGINAL SAVE (unchanged)
            torch.save(model_dir.state_dict(), CKPT_DIR)

            # NEW: local fast save
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_dir.state_dict(),
                'optimizer_state_dict': opt_dir.state_dict(),
                'loss': best_val_mse_dir,
            }, local_save_path_dir)

            saved = '✅'

        print(f"{epoch:>6}  {train_loss:>9.4f}  {m['mse']:>9.2f}  "
              f"{m['mean_dt']:>7.2f}%  {m['slack_mse']:>10.4f}  {m['slack_mae']:>10.4f}  "
              f"{m['spearman']:>7.4f}  {saved}")

print(f'\n✅ Directed done. Best val MSE: {best_val_mse_dir:.4f}')

# NEW: copy once to Drive
shutil.copy(local_save_path_dir, drive_save_path_dir)
print("✅ Directed model copied to Drive!")

 Epoch    Train L    Val MSE   Val ΔT%   Slack MSE   Slack MAE    Val ρ  Saved
------------------------------------------------------------------------------------------
    10    54.7312     106.80    29.00%      0.0037      0.0182  -0.0378  ✅
    20    44.4663      72.49    20.44%      0.0037      0.0194   0.0376  ✅
    30     9.7656       8.49     5.84%      0.0035      0.0177   0.0400  ✅
    40     8.4221      10.06     4.82%      0.0044      0.0253   0.0372  
    50     7.5909      10.85     4.37%      0.0035      0.0162   0.0411  
    60     6.7458      10.72     3.97%      0.0035      0.0160   0.0452  
    70     5.9651      12.22     3.76%      0.0035      0.0157   0.0462  
    80     5.2791      12.67     4.38%      0.0035      0.0152   0.0510  

✅ Directed done. Best val MSE: 8.4950
✅ Directed model copied to Drive!


## 7. Final evaluation — held-out test set

Load best checkpoints, run on test set.  Also compute 1F1B analytic baseline
**on the same test split** for reference.

**1F1B formula:**  step time ≈ sum of all compute-node durations (column 0 of x).
This is the sum of raw op durations, i.e. the serial lower bound.


In [12]:
# ── load best checkpoints ────────────────────────────────────────────────────
model_bidir.load_state_dict(torch.load(CKPT_BIDIR, map_location=DEVICE))
model_dir.load_state_dict(torch.load(CKPT_DIR,   map_location=DEVICE))

bidir_m = evaluate(model_bidir, test_graphs)
dir_m   = evaluate(model_dir,   test_graphs)

# ── 1F1B analytic on same test set ───────────────────────────────────────────
# Formula: predicted step time = sum of all compute-node raw durations.
def predict_1f1b(g):
    return g['compute'].x[:, 0].sum().item()

true_times  = [g.y.item() for g in test_graphs]
f1b_preds   = [predict_1f1b(g) for g in test_graphs]
f1b_mse     = mean_squared_error(true_times, f1b_preds)
f1b_deltas  = [delta_t_pct(p, t) for p, t in zip(f1b_preds, true_times)]

# ── comparison table ─────────────────────────────────────────────────────────
print(f"\n{'='*100}")
print(f"{'Model':<25} {'MSE':>10} {'Mean ΔT%':>10} {'Median ΔT%':>12} {'Slack MSE':>12} {'Slack MAE':>12} {'Spearman ρ':>12}")
print(f"{'-'*100}")
print(f"{'1F1B Analytic (ref)':<25} {f1b_mse:>10.4f} "      f"{np.mean(f1b_deltas):>10.2f} {np.median(f1b_deltas):>12.2f} "
      f"{'—':>12} {'—':>12} {'—':>12}")
print(f"{'HeteroGAT (bidir)':<25} {bidir_m['mse']:>10.4f} "      f"{bidir_m['mean_dt']:>10.2f} {bidir_m['median_dt']:>12.2f} "
      f"{bidir_m['slack_mse']:>12.4f} {bidir_m['slack_mae']:>12.4f} "      f"{bidir_m['spearman']:>12.4f}")
print(f"{'HeteroGAT (directed)':<25} {dir_m['mse']:>10.4f} "      f"{dir_m['mean_dt']:>10.2f} {dir_m['median_dt']:>12.2f} "
      f"{dir_m['slack_mse']:>12.4f} {dir_m['slack_mae']:>12.4f} "      f"{dir_m['spearman']:>12.4f}")
print(f"{'='*100}")

# ── acceptance criteria check (updated for slack regression) ───────────────
print("\n📋 Acceptance criteria (updated for slack regression):")
rho_ok = dir_m['spearman'] > 0.70
dt_bidir_ok = bidir_m['mean_dt'] < 5.0
dt_dir_ok   = dir_m['mean_dt']   < 5.0
slack_mse_ok = dir_m['slack_mse'] < 1.0  # New criterion: good slack prediction
print(f"  Directed Spearman ρ > 0.70           : {dir_m['spearman']:.4f}  {'✅' if rho_ok else '❌'}")
print(f"  Bidir ΔT% < 5%%                       : {bidir_m['mean_dt']:.2f}%%  {'✅' if dt_bidir_ok else '❌'}")
print(f"  Directed ΔT% < 5%%                    : {dir_m['mean_dt']:.2f}%%  {'✅' if dt_dir_ok else '❌'}")
print(f"  Directed Slack MSE < 1.0ms            : {dir_m['slack_mse']:.4f}  {'✅' if slack_mse_ok else '❌'}")


Model                            MSE   Mean ΔT%   Median ΔT%    Slack MSE    Slack MAE   Spearman ρ
----------------------------------------------------------------------------------------------------
1F1B Analytic (ref)           0.0000       0.00         0.00            —            —            —
HeteroGAT (bidir)             4.7643       3.96         2.47       0.0054       0.0161       0.0853
HeteroGAT (directed)         12.6719       6.46         4.30       0.0052       0.0180       0.0335

📋 Acceptance criteria (updated for slack regression):
  Directed Spearman ρ > 0.70           : 0.0335  ❌
  Bidir ΔT% < 5%%                       : 3.96%%  ✅
  Directed ΔT% < 5%%                    : 6.46%%  ❌
  Directed Slack MSE < 1.0ms            : 0.0052  ✅


## 8. Bottleneck inspection — top-5 critical nodes

For one test graph, show which nodes the **directed model** scores highest.
This is the output the Sprint 6 GraphRAG loop will consume.


In [13]:
model_dir.eval()
g = test_graphs[0].to(DEVICE)

with torch.no_grad():
    node_slack, step_pred = model_dir(g.x_dict, g.edge_index_dict)

pred_slack = node_slack.cpu().numpy()
true_slack = g['compute'].y.cpu().numpy()
durs   = g['compute'].x[:, 0].cpu().numpy()    # raw duration_ms

# Show top-5 nodes with SMALLEST predicted slack (most critical)
top5_critical = np.argsort(pred_slack)[:5]  # Smallest slack = most critical

rho_val, _ = spearmanr(pred_slack, true_slack)

print("Top-5 most critical nodes (smallest predicted slack):")
print(f"{'Node':>6}  {'Pred Slack':>12}  {'True Slack':>12}  {'Dur (ms)':>10}  {'Binary':>7}")
print('-' * 65)
for i in top5_critical:
    binary_label = g['compute'].y_binary[i].cpu().item()
    print(f"{i:>6}  {pred_slack[i]:>12.4f}  {true_slack[i]:>12.4f}  {durs[i]:>10.3f}  {int(binary_label):>7}")

print(f"\nSpearman ρ on this graph (slack): {rho_val:.4f}")
print(f"Predicted step time      : {step_pred.item():.2f} ms")
print(f"True step time           : {g.y.item():.2f} ms")
print(f"ΔT%                      : {delta_t_pct(step_pred.item(), g.y.item()):.2f}%")

# Show overall statistics
print(f"\nOverall statistics:")
print(f"  Mean true slack: {true_slack.mean():.4f} ms")
print(f"  Mean pred slack: {pred_slack.mean():.4f} ms")
print(f"  Std true slack:  {true_slack.std():.4f} ms")
print(f"  Std pred slack:  {pred_slack.std():.4f} ms")

Top-5 most critical nodes (smallest predicted slack):
  Node    Pred Slack    True Slack    Dur (ms)   Binary
-----------------------------------------------------------------
   152        0.0128        0.0000       0.036        1
   146        0.0128        0.0000       0.010        1
   148        0.0128        0.0000       0.005        1
   153        0.0128        0.0018       0.034        0
   149        0.0128        0.0035       0.001        0

Spearman ρ on this graph (slack): 0.0222
Predicted step time      : 19.04 ms
True step time           : 17.68 ms
ΔT%                      : 7.69%

Overall statistics:
  Mean true slack: 0.0142 ms
  Mean pred slack: 0.0140 ms
  Std true slack:  0.0176 ms
  Std pred slack:  0.0029 ms


## 9. Save checkpoints to HuggingFace

Uncomment and run once training is complete so weights are persisted across sessions.
Both checkpoints go to the same HF dataset repo as the graphs.


In [14]:
# from huggingface_hub import HfApi, login
# login(token="YOUR_HF_TOKEN_HERE")   # or use Kaggle secrets / Colab secrets
# api = HfApi()
#
# for ckpt in [CKPT_BIDIR, CKPT_DIR]:
#     api.upload_file(
#         path_or_fileobj=ckpt,
#         path_in_repo=f"checkpoints/{os.path.basename(ckpt)}",
#         repo_id=REPO,
#         repo_type='dataset',
#     )
#     print(f'✅ Uploaded {ckpt}')
